### 휘발성 메모리로 일반 변수에 대화 내용 저장하기
- 대화 기록은 사용자가 대화를 진행하는 동안만 유지하며
- 세션이 종료되면 사라지는 휘발성 저장 방식

In [1]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.output_parsers import StrOutputParser

import os

from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langchain_teddynote import logging


load_dotenv()
logging.langsmith("test0914")


print("OpenAI 키 로드됨 : ", bool(os.getenv("OPENAI_API_KEY")))
print("LangSmith 키 로드됨 : ", bool(os.getenv("LANGSMITH_API_KEY")))
print("LangSmith 프로젝트 : ", os.getenv("LANGSMITH_PROJECT"))

LangSmith 추적을 시작합니다.
[프로젝트명]
test0914
OpenAI 키 로드됨 :  True
LangSmith 키 로드됨 :  True
LangSmith 프로젝트 :  test0914


In [8]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 Question-Answering 챗봇입니다. 주어진 질문에 대한 답변을 제공해 주세요."),

        # 대화 기록용 key 인 chat_history는 가급적 변경 없이 사용하세요!
        MessagesPlaceholder(variable_name="chat_history"),
        ("human","#Question:\n{question}")
    ]
)

llm = ChatOpenAI(model="gpt-4o")

chain = prompt | llm | StrOutputParser()

In [9]:
store = {}

# 세션 ID를 기반으로 세션 기록을 가져오는 함수
def get_session_history(session_ids):
    print(f"[대화 세션ID]: { session_ids}")
    if session_ids not in store:  # 세션 아이디가 store에 없는 경우
        store[session_ids] = ChatMessageHistory() # 새로운 ChatMessageHistory 객체를 생성하여 store에 저장

    return store[session_ids] # 해당 세션 ID에 대한 세션 기록 반환

In [10]:
chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,   # 세션 기록을 가져오는 함수
    input_messages_key="question",  # 사용자의 질문이 템플릿 변수에 들어갈 키
    history_messages_key="chat_history"  # 기록 메시지의 키
)

In [11]:
chain_with_history.invoke(
    {"question":"나의 이름은 테디입니다."},
    config={"configurable": {"session_id":"abc123"}}
)

[대화 세션ID]: abc123


'안녕하세요, 테디! 어떻게 도와드릴까요?'

In [12]:
chain_with_history.invoke(
    {"question":"나의 이름은 테디입니다."},
    config={"configurable": {"session_id":"abc1234"}}
)

[대화 세션ID]: abc1234


'안녕하세요, 테디님! 어떻게 도와드릴까요?'

In [15]:
chain_with_history.invoke(
    {"question": "내 이름이 뭐라고 했지?"},
    config={"configurable": {"session_id": "abcd1234"}},
)

[대화 세션ID]: abcd1234


'죄송하지만, 이전 대화 내용을 확인할 수 없어서 당신의 이름을 기억할 수 없습니다. 자기소개 부탁드립니다!'